# Practica: Coeficiente Tau de Kendall (2.1.3)

**Asignatura:** Analisis y visualizacion de datos (GAD-2401)  
**Tema:** 2.1.3 – Coeficiente Tau de Kendall (tau)

## Objetivos

1. Contar pares concordantes y discordantes.
2. Calcular tau de forma manual (logica) y con scipy.
3. Comparar Pearson, Spearman y Kendall.
4. Verificar los ejercicios del cuaderno.
5. Aplicar a rankings y a un dataset real.

> **Importante:** Resuelve primero los ejercicios de `ejercicios/ejercicios-manuales.html` en tu cuaderno.


## 1. Importacion de librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
print("Librerias cargadas correctamente.")

## 2. Funcion: conteo de pares y tau paso a paso

In [ ]:
def kendall_manual_detalle(x, y, nombre_x="X", nombre_y="Y"):
    """
    Cuenta pares concordantes/discordantes y calcula tau-a.
    Tambien reporta tau-b de scipy.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    C, D, ties = 0, 0, 0
    pares = []

    for i, j in combinations(range(n), 2):
        dx = x[j] - x[i]
        dy = y[j] - y[i]
        prod = dx * dy
        if prod > 0:
            tipo = "C"
            C += 1
        elif prod < 0:
            tipo = "D"
            D += 1
        else:
            tipo = "empate"
            ties += 1
        pares.append((i+1, j+1, x[i], x[j], y[i], y[j], tipo))

    total = n * (n - 1) / 2
    tau_a = (C - D) / total if total else np.nan
    tau_b, p_valor = stats.kendalltau(x, y)

    print("=" * 70)
    print(f"Kendall entre {nombre_x} y {nombre_y}")
    print("=" * 70)
    print(f"n = {n}  |  pares posibles = {int(total)}")
    print(f"{'i':>3} {'j':>3} {'xi':>8} {'xj':>8} {'yi':>8} {'yj':>8} {'tipo':>8}")
    print("-" * 70)
    for (i, j, xi, xj, yi, yj, tipo) in pares:
        print(f"{i:>3} {j:>3} {xi:>8.2f} {xj:>8.2f} {yi:>8.2f} {yj:>8.2f} {tipo:>8}")
    print("-" * 70)
    print(f"C (concordantes) = {C}")
    print(f"D (discordantes) = {D}")
    print(f"Empates          = {ties}")
    print(f"tau-a (manual)   = {tau_a:.4f}")
    print(f"tau-b (scipy)    = {tau_b:.4f}")
    print(f"p-valor          = {p_valor:.6e}")
    print("=" * 70)
    return tau_b, p_valor

## 3. Verificacion del Ejercicio 1

In [ ]:
x1 = [1, 2, 3, 4]
y1 = [2, 1, 4, 3]
tau1, p1 = kendall_manual_detalle(x1, y1, "X", "Y")

## 4. Verificacion del Ejercicio 2 (horas vs calificacion)

In [ ]:
x2 = [1, 2, 4, 5, 7]
y2 = [55, 60, 75, 80, 90]
tau2, p2 = kendall_manual_detalle(x2, y2, "Horas", "Calificacion")

r2, _ = stats.pearsonr(x2, y2)
rho2, _ = stats.spearmanr(x2, y2)
print(f"\nComparacion: Pearson r={r2:.4f} | Spearman rho={rho2:.4f} | Kendall tau={tau2:.4f}")

## 5. Verificacion del Ejercicio 3 (relacion negativa)

In [ ]:
x3 = [40, 50, 60, 70, 80]
y3 = [90, 75, 60, 50, 35]
tau3, p3 = kendall_manual_detalle(x3, y3, "Velocidad", "Tiempo")

r3, _ = stats.pearsonr(x3, y3)
rho3, _ = stats.spearmanr(x3, y3)
print(f"\nComparacion: Pearson r={r3:.4f} | Spearman rho={rho3:.4f} | Kendall tau={tau3:.4f}")

## 6. Ejercicio 5 – Rankings de docentes

In [ ]:
rank_A = [1, 2, 3, 4, 5]
rank_B = [1, 3, 2, 5, 4]
tau5, p5 = kendall_manual_detalle(rank_A, rank_B, "Docente A", "Docente B")
print("\nValores altos de tau indican mayor concordancia entre rankings.")

## 7. Comparacion de los tres coeficientes (dataset tips)

In [ ]:
tips = sns.load_dataset("tips")
x = tips["total_bill"]
y = tips["tip"]

r, pr = stats.pearsonr(x, y)
rho, prho = stats.spearmanr(x, y)
tau, ptau = stats.kendalltau(x, y)

print("total_bill vs tip")
print(f"  Pearson  r   = {r:.4f}  (p={pr:.2e})")
print(f"  Spearman rho = {rho:.4f}  (p={prho:.2e})")
print(f"  Kendall  tau = {tau:.4f}  (p={ptau:.2e})")
print("\nNota: |tau| suele ser menor que |rho| para la misma asociacion.")

num = tips.select_dtypes(include="number")
corr_k = num.corr(method="kendall")
display(corr_k.round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(corr_k, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, fmt=".3f")
plt.title("Mapa de calor – Kendall (tips)")
plt.tight_layout()
plt.show()

## 8. Ejercicios adicionales

### Ejercicio A
Con `tips`, calcula los tres coeficientes para otro par de variables y comenta diferencias.

### Ejercicio B
Inventa 5 pares (x, y) con asociacion monotona moderada, cuenta C y D a mano en el cuaderno
y verifica aqui con la funcion.

### Ejercicio C (reflexion)
1. Por que |tau| suele ser menor que |rho|?
2. En que situacion preferirias Kendall frente a Spearman?
3. Explica tau = 0.4 en terminos de probabilidad de concordancia.


### Espacio para Ejercicio A

In [ ]:
# Tu codigo aqui


### Espacio para Ejercicio B

In [ ]:
# Tu codigo aqui


### Espacio para Ejercicio C

Respuestas:

1. 
2. 
3. 


## 9. Conclusiones

- Tau se basa en pares **concordantes** y **discordantes**.
- Es no parametrico, robusto y muy util con muestras pequenas o empates.
- tau-b (scipy) ajusta empates; en ejercicios sin empates tau-a = tau-b.
- Siempre reportar tau, p-valor y n; visualizar antes de interpretar.

**Fin de la practica del subtema 2.1.3**
